In [1]:
import numpy as np
import pandas as pd

np.random.seed(123)

n = 600

income = np.random.normal(65000, 18000, n).clip(20000, 150000)
loan_amount = np.random.normal(18000, 7000, n).clip(2000, 45000)
credit_score = np.random.normal(680, 60, n).clip(500, 850)
debt_to_income = np.random.normal(0.32, 0.12, n).clip(0.05, 0.80)
past_due_count = np.random.poisson(0.8, n).clip(0, 6)
employment_length = np.random.normal(6, 3, n).clip(0, 20)

# True log-odds used only to generate a realistic binary outcome
lin = (
    -0.00005 * income
    + 0.00009 * loan_amount
    - 0.015 * (credit_score - 680)
    + 4.5 * (debt_to_income - 0.32)
    + 0.8 * past_due_count
    - 0.12 * (employment_length - 6)
    - 1.5
)

p_default = 1 / (1 + np.exp(-lin))
default = np.random.binomial(1, p_default)

df = pd.DataFrame({
    "income": income,
    "loan_amount": loan_amount,
    "credit_score": credit_score,
    "debt_to_income": debt_to_income,
    "past_due_count": past_due_count,
    "employment_length": employment_length,
    "default": default
})

summary_raw = df.agg(["min", "max", "mean", "std"])
print(summary_raw)
print("\nDefault rate:", df["default"].mean())

             income   loan_amount  credit_score  debt_to_income  \
min    20000.000000   2000.000000    500.000000        0.050000   
max   118255.258026  43001.054526    850.000000        0.630441   
mean   64790.181562  17696.526848    679.279128        0.328572   
std    17935.534400   6874.661936     57.016183        0.112453   

      past_due_count  employment_length   default  
min         0.000000           0.000000  0.000000  
max         5.000000          15.152265  1.000000  
mean        0.813333           6.066754  0.156667  
std         0.879267           2.975982  0.363789  

Default rate: 0.15666666666666668


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

feature_names = [
    "income",
    "loan_amount",
    "credit_score",
    "debt_to_income",
    "past_due_count",
    "employment_length"
]

X = df[feature_names]
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=123, stratify=y
)

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

linear_pred = linear_model.predict(X_test)

print("Range of actual response y:", y_test.min(), "to", y_test.max())
print("Range of linear model predictions:", linear_pred.min(), "to", linear_pred.max())

Range of actual response y: 0 to 1
Range of linear model predictions: -0.41071979888938404 to 0.817336762993793


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

logit = LogisticRegression(max_iter=3000)
logit.fit(X_train, y_train)

y_prob = logit.predict_proba(X_test)[:, 1]
y_pred = logit.predict(X_test)

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print("Accuracy:", acc)
print("AUC:", auc)
print("Confusion matrix:\n", cm)

coef_table = pd.DataFrame({
    "Predictor": feature_names,
    "Coefficient": logit.coef_[0]
})
print("\nCoefficient table:")
print(coef_table)

Accuracy: 0.9066666666666666
AUC: 0.9079082505991098
Confusion matrix:
 [[123   4]
 [ 10  13]]

Coefficient table:
           Predictor  Coefficient
0             income    -0.000052
1        loan_amount     0.000084
2       credit_score    -0.020114
3     debt_to_income     1.520481
4     past_due_count     1.098619
5  employment_length    -0.188706


In [5]:
from sklearn.metrics import confusion_matrix

for threshold in [0.50, 0.30, 0.20]:
    y_pred_t = (y_prob >= threshold).astype(int)
    cm_t = confusion_matrix(y_test, y_pred_t)
    tn, fp, fn, tp = cm_t.ravel()

    accuracy = (tn + tp) / cm_t.sum()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)

    print(f"\nThreshold = {threshold:.2f}")
    print("Confusion matrix:\n", cm_t)
    print("Accuracy:", round(accuracy, 4))
    print("Sensitivity:", round(sensitivity, 4))
    print("Specificity:", round(specificity, 4))


Threshold = 0.50
Confusion matrix:
 [[123   4]
 [ 10  13]]
Accuracy: 0.9067
Sensitivity: 0.5652
Specificity: 0.9685

Threshold = 0.30
Confusion matrix:
 [[117  10]
 [  6  17]]
Accuracy: 0.8933
Sensitivity: 0.7391
Specificity: 0.9213

Threshold = 0.20
Confusion matrix:
 [[102  25]
 [  6  17]]
Accuracy: 0.7933
Sensitivity: 0.7391
Specificity: 0.8031


In [6]:
cases = pd.DataFrame([
    {
        "income": 78000,
        "loan_amount": 12000,
        "credit_score": 735,
        "debt_to_income": 0.22,
        "past_due_count": 0,
        "employment_length": 8
    },
    {
        "income": 45000,
        "loan_amount": 18000,
        "credit_score": 620,
        "debt_to_income": 0.42,
        "past_due_count": 2,
        "employment_length": 5
    },
    {
        "income": 41000,
        "loan_amount": 28000,
        "credit_score": 590,
        "debt_to_income": 0.53,
        "past_due_count": 3,
        "employment_length": 2
    }
])

case_probs = logit.predict_proba(cases)[:, 1]

results = cases.copy()
results["predicted_default_probability"] = case_probs
print(results)

   income  loan_amount  credit_score  debt_to_income  past_due_count  \
0   78000        12000           735            0.22               0   
1   45000        18000           620            0.42               2   
2   41000        28000           590            0.53               3   

   employment_length  predicted_default_probability  
0                  8                       0.001833  
1                  5                       0.785604  
2                  2                       0.991711  
